# Global Shipping Chokepoints — 04: Export a Light Version for the Web Map

The full traffic grid (3,040,763 rows) is too big and too fine-grained to put in a
browser map. This notebook re-aggregates it to a coarser 0.5-degree grid (still real
data, just fewer, bigger cells — good enough for a world-scale heat map) and prints it
as compact JSON directly in the cell output, along with the final chokepoint comparison
table, so both can be copied straight out of the downloaded notebook without needing to
download a separate file from a volume.

In [0]:
from pyspark.sql import functions as F

GRID_DEGREES = 0.5  # heat-map resolution: coarse enough for a world map, still built from real data

traffic = spark.table("workspace.global_shipping.traffic_density_grid")

coarse = (
    traffic
    .withColumn("grid_lat", F.floor(F.col("lat") / GRID_DEGREES) * GRID_DEGREES)
    .withColumn("grid_lon", F.floor(F.col("lon") / GRID_DEGREES) * GRID_DEGREES)
    .groupBy("grid_lat", "grid_lon")
    .agg(F.sum("traffic_density").alias("traffic_density"))
)

print(f"Coarse grid cells: {coarse.count():,} (down from {traffic.count():,} at native 0.05-degree resolution)")

Coarse grid cells: 113,482 (down from 3,040,763 at native 0.05-degree resolution)


## Print as compact JSON

Rounds density to the nearest thousand and drops the smallest 50% of cells (the
faintest, least visually meaningful traffic) to keep the printed payload small — this
is purely a size cut for the visualization, the full real data stays in the Delta table
untouched for anyone who wants the complete grid.

In [0]:
import json

pdf = coarse.toPandas()
median_density = pdf["traffic_density"].median()
pdf = pdf[pdf["traffic_density"] >= median_density].copy()
pdf["traffic_density"] = (pdf["traffic_density"] / 1000).round().astype("int64")  # store in thousands

records = pdf.round({"grid_lat": 2, "grid_lon": 2}).to_dict(orient="records")
print(f"Cells in export (>= median density): {len(records):,}")
print(json.dumps(records, separators=(",", ":")))

Cells in export (>= median density): 56,741


[{"grid_lat":84.5,"grid_lon":134.0,"traffic_density":964482},{"grid_lat":80.0,"grid_lon":15.5,"traffic_density":1045517},{"grid_lat":80.0,"grid_lon":27.0,"traffic_density":2188972},{"grid_lat":78.5,"grid_lon":-2.5,"traffic_density":1262516},{"grid_lat":78.5,"grid_lon":7.0,"traffic_density":1568227},{"grid_lat":78.5,"grid_lon":46.0,"traffic_density":2188618},{"grid_lat":78.0,"grid_lon":37.5,"traffic_density":1207518},{"grid_lat":78.0,"grid_lon":128.0,"traffic_density":636627},{"grid_lat":78.0,"grid_lon":-88.0,"traffic_density":2188425},{"grid_lat":78.0,"grid_lon":90.5,"traffic_density":2506360},{"grid_lat":77.5,"grid_lon":35.0,"traffic_density":1904872},{"grid_lat":77.5,"grid_lon":15.5,"traffic_density":5211295},{"grid_lat":77.5,"grid_lon":16.5,"traffic_density":908703},{"grid_lat":77.5,"grid_lon":120.0,"traffic_density":2188092},{"grid_lat":77.5,"grid_lon":112.0,"traffic_density":1214530},{"grid_lat":77.0,"grid_lon":63.0,"traffic_density":9006292},{"grid_lat":77.0,"grid_lon":67.5,"traf

## Print the final chokepoint comparison table as JSON too

In [0]:
final = spark.table("workspace.global_shipping.chokepoints_final_comparison").toPandas()
print(json.dumps(final.to_dict(orient="records"), default=str, separators=(",", ":")))

[{"chokepoint":"Strait of Malacca","total_traffic":1286868516300,"n_cells_with_traffic":5553,"lat_min":1.0,"lat_max":6.5,"lon_min":98.0,"lon_max":104.5,"share_of_ranked_total":0.13952402891848564,"center_lat":3.75,"center_lon":101.25,"nearest_port":"TELUK ANSON","nearest_port_country":"MY","portwatch_name":"Malacca Strait","portwatch_days_available":366,"avg_daily_vessels":230.21857923497268,"avg_daily_capacity":8660301.521857923,"capacity_share":0.3926905347306774},{"chokepoint":"Strait of Gibraltar","total_traffic":155948076200,"n_cells_with_traffic":93,"lat_min":35.7,"lat_max":36.2,"lon_min":-5.9,"lon_max":-5.2,"share_of_ranked_total":0.016908101812973853,"center_lat":35.95,"center_lon":-5.550000000000001,"nearest_port":"TANGIER-MEDITERRANEAN","nearest_port_country":"MA","portwatch_name":"Gibraltar Strait","portwatch_days_available":366,"avg_daily_vessels":133.69945355191257,"avg_daily_capacity":3521643.6174863386,"capacity_share":0.1596845227375993},{"chokepoint":"Strait of Dover",